# 03 — XLA compilation simulation: HLO, cache hits, silent recompiles

**Learning goal.** Understand what XLA actually does when you call `jax.jit` (or TF `tf.function`, or `torch_xla`'s compile): it builds an HLO graph, hashes it, and either compiles or pulls from a cache. Shape changes invalidate the cache — this is the classic "silent recompile" footgun.

**What you'll observe.**
- A model graph lowered to a fake HLO module.
- The op count by kind and the total FLOPS reported by the module.
- A cache **miss** on first compile and a cache **hit** on the second compile of the same graph.
- A cache **miss** caused by changing only `batch_size` — the dreaded silent recompile.

**Knobs to tune.**
- `hidden_size`, `num_layers` — change the HLO op count and FLOPS.
- `batch_size` on the second compile — triggers a fresh compile.

**Failure modes to watch.**
- In real workloads, dynamic shapes (variable batch size, variable seq_len) cause repeated compiles. Each compile can take seconds to minutes for big models.
- If you see compile time dominating your wall clock, look first at shape stability.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Build a tiny model graph and lower to HLO

In [ ]:
from cloud_tpu_lab.src.model_examples.tiny_mlp_jax import build_tiny_mlp_graph
from cloud_tpu_lab.src.xla_sim.lowering import lower_to_hlo

graph = build_tiny_mlp_graph(batch_size=8, input_dim=64, hidden_size=128,
                             output_dim=10, num_layers=3, dtype="bf16")
module = lower_to_hlo(graph)

print("HloModule name:", module.name)
print("entry op:", module.entry_op_id)
print("n ops:", len(module.ops))
print("op count by kind:", module.op_count_by_kind())
print("total FLOPS:", module.total_flops())
print("total bytes moved:", module.total_bytes_moved())

## First compile — cache MISS

The compile cache is empty. The simulator returns a non-zero `compile_time_s` and a fresh `executable_id`.

In [ ]:
from cloud_tpu_lab.src.xla_sim.compile_cache import CompileCache, compile_hlo

cache = CompileCache()
r1 = compile_hlo(module, cache, mesh_shape=(1,), dtype="bf16")
print("compile_time_s:", r1.compile_time_s)
print("cache_hit:", r1.cache_hit)
print("executable_id:", r1.executable_id)
print("cache stats:", cache.stats())

## Second compile — cache HIT

Same module, same mesh, same dtype. The key matches; we get the cached executable for ~0 cost.

In [ ]:
r2 = compile_hlo(module, cache, mesh_shape=(1,), dtype="bf16")
print("compile_time_s:", r2.compile_time_s)
print("cache_hit:", r2.cache_hit)
print("executable_id same as before?", r2.executable_id == r1.executable_id)
print("cache stats:", cache.stats())

## Silent recompile — shape change invalidates the cache

Change only the batch size. The shapes inside the HLO ops change, so the cache key changes, so we get a fresh compile.

In [ ]:
graph_b16 = build_tiny_mlp_graph(batch_size=16, input_dim=64, hidden_size=128,
                                 output_dim=10, num_layers=3, dtype="bf16")
module_b16 = lower_to_hlo(graph_b16)

r3 = compile_hlo(module_b16, cache, mesh_shape=(1,), dtype="bf16")
print("compile_time_s:", r3.compile_time_s)
print("cache_hit:", r3.cache_hit)
print("executable_id:", r3.executable_id)
print("cache stats:", cache.stats())

## Plot: cumulative compile cost across a "bad" loop

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cache2 = CompileCache()
batch_sizes = [4, 8, 4, 16, 8, 32, 4, 8]   # mixed; some repeat, some new
compile_times = []
cumulative = 0.0
for b in batch_sizes:
    g = build_tiny_mlp_graph(batch_size=b, input_dim=64, hidden_size=128,
                             output_dim=10, num_layers=3, dtype="bf16")
    m = lower_to_hlo(g)
    r = compile_hlo(m, cache2, mesh_shape=(1,), dtype="bf16")
    cumulative += r.compile_time_s
    compile_times.append(cumulative)
    print(f"batch={b:>3}  hit={r.cache_hit}  compile_s={r.compile_time_s:.4f}  cum={cumulative:.4f}")

fig = plt.figure(figsize=(7, 4))
plt.plot(range(len(batch_sizes)), compile_times, marker="o")
plt.xticks(range(len(batch_sizes)), [str(b) for b in batch_sizes])
plt.xlabel("call # (label = batch_size)")
plt.ylabel("cumulative simulated compile time (s)")
plt.title("Silent recompiles: each new shape = full compile")
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb03_compile_cache.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)
print("final cache stats:", cache2.stats())

## Takeaways

- The cache key in this simulator hashes ops, shapes, dtypes, and mesh shape. Real XLA hashes more (sharding annotations, fusion choices), but the principle is the same: **anything in the key that changes forces a fresh compile.**
- In production: pad inputs to fixed shapes, or use bucketing — never let `batch_size` change every call.

## Exercises

1. Re-run the second cell with `mesh_shape=(2,)`. Does the key change? Cache hit or miss?
2. Build a `transformer`-style graph with varying `seq_len`. Show that each new `seq_len` causes a recompile.
3. Add `precision="fp32"` and `precision="bf16"` runs with otherwise identical configs. Are they the same cache entry?
4. Plot `compile_time_s` against `total_flops` for layer counts 1..8. Is it linear in FLOPS? (Look at `compile_hlo`'s formula.)
5. Devise a real-world scenario in which the cache *appears* to work but is silently invalidated every step. How would you detect it from logs?